In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATASET_ID = 3

ds_name = f"ds{DATASET_ID}" 
file_suffix = f"{DATASET_ID:02d}"

PROJECT_ROOT = Path.cwd()
ART_DIR = PROJECT_ROOT / "artifacts"
FIG_DIR = ART_DIR / "figures"
LABEL_DIR = ART_DIR / "labels"
DATA_DIR = PROJECT_ROOT / "data"

input_filename = DATA_DIR / f"S07-hw-dataset-{file_suffix}.csv"
df = pd.read_csv(input_filename)

print(f"Загружен датасет: {input_filename}")
print(f"Идентификатор вывода: {ds_name}")
print(df.head())
print(df.shape)
print(df.describe())
print(df.info())
print(df.isnull().sum)

Загружен датасет: c:\Users\dimas\Desktop\Learning\Add\AI agent\ai_agent_memerea\homeworks\HW07\data\S07-hw-dataset-03.csv
Идентификатор вывода: ds3
   sample_id        x1        x2    f_corr   f_noise
0          0 -2.710470  4.997107 -1.015703  0.718508
1          1  8.730238 -8.787416  3.953063 -1.105349
2          2 -1.079600 -2.558708  0.976628 -3.605776
3          3  6.854042  1.560181  1.760614 -1.230946
4          4  9.963812 -8.869921  2.966583  0.915899
(15000, 5)
          sample_id            x1            x2        f_corr       f_noise
count  15000.000000  15000.000000  15000.000000  15000.000000  15000.000000
mean    7499.500000      1.246296      1.033764      0.212776     -0.027067
std     4330.271354      4.592421      4.710791      1.530017      2.506375
min        0.000000     -9.995585     -9.980853     -5.212038     -8.785884
25%     3749.750000     -1.782144     -2.666393     -0.966224     -1.731128
50%     7499.500000      0.664226      1.831257      0.296508     -

In [89]:
sample_id = df["sample_id"]
X_raw = df.drop(columns=["sample_id", "f_corr", "f_noise"])

In [90]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

num_cols = X_raw.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols)
    ]
)

X = preprocessor.fit_transform(X_raw)


In [91]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_values = range(2, 21)
sil_scores = []

kmeans_models = {}

for k in k_values:
    model_km = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )
    labels_km = model_km.fit_predict(X)
    score = silhouette_score(X, labels_km)
    sil_scores.append(score)
    kmeans_models[k] = (model_km, labels_km)

In [92]:
# График silhouette vs k
import matplotlib.pyplot as plt

plt.figure()
plt.plot(k_values, sil_scores, marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette score")
plt.title(f"KMeans: silhouette vs k ({ds_name})")
save_path = FIG_DIR / f"{ds_name}_kmeans_silhouette_vs_k.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()

best_k = k_values[int(np.argmax(sil_scores))]
best_km_model, best_km_labels = kmeans_models[best_k]
print("Лучший k по silhouette:", best_k)

Лучший k по silhouette: 5


In [93]:
from sklearn.cluster import DBSCAN

eps_values = [0.2, 0.4, 0.6, 0.8, 1.0]
dbscan_results = []

for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels_db = db.fit_predict(X)

    noise_share = np.mean(labels_db == -1)

    # считаем количество «реальных» кластеров (без шума)
    unique_labels = set(labels_db[labels_db != -1])

    if len(unique_labels) < 2:       # <-- ключевая проверка
        sil = np.nan
    else:
        mask = labels_db != -1
        sil = silhouette_score(X[mask], labels_db[mask])

    dbscan_results.append((eps, sil, noise_share))

df_db = pd.DataFrame(
    dbscan_results, 
    columns=["eps", "silhouette_non_noise", "noise_share"]
)
print(df_db)

   eps  silhouette_non_noise  noise_share
0  0.2             -0.043439     0.002533
1  0.4                   NaN     0.000000
2  0.6                   NaN     0.000000
3  0.8                   NaN     0.000000
4  1.0                   NaN     0.000000


In [94]:
# График
plt.figure()
plt.plot(df_db["eps"], df_db["silhouette_non_noise"], marker="o")
plt.xlabel("eps")
plt.ylabel("Silhouette (без шума)")
plt.title(f"DBSCAN: silhouette vs eps ({ds_name})")
save_path = FIG_DIR / f"{ds_name}_dbscan_silhouette_vs_eps.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()

best_eps = df_db.loc[df_db["silhouette_non_noise"].idxmax(), "eps"]
best_db = DBSCAN(eps=best_eps, min_samples=5)
best_db_labels = best_db.fit_predict(X)

print("Лучший eps:", best_eps)
print("Доля шума:", np.mean(best_db_labels == -1))

Лучший eps: 0.2
Доля шума: 0.002533333333333333


In [95]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# --- KMeans ---
sil_km = silhouette_score(X, best_km_labels)
db_km = davies_bouldin_score(X, best_km_labels)
ch_km = calinski_harabasz_score(X, best_km_labels)

# --- DBSCAN (без шума) ---
mask = best_db_labels != -1
sil_db = silhouette_score(X[mask], best_db_labels[mask])
db_db = davies_bouldin_score(X[mask], best_db_labels[mask])
ch_db = calinski_harabasz_score(X[mask], best_db_labels[mask])
noise_share = np.mean(best_db_labels == -1)


In [96]:
import json

metrics_summary = {
    ds_name: {  # <--- Здесь теперь используется переменная "ds3" или "ds2"
        "KMeans": {
            "k": best_k,
            "silhouette": float(sil_km),
            "davies_bouldin": float(db_km),
            "calinski_harabasz": float(ch_km)
        },
        "DBSCAN": {
            "eps": best_eps,
            "silhouette_non_noise": float(sil_db),
            "davies_bouldin_non_noise": float(db_db),
            "calinski_harabasz_non_noise": float(ch_db),
            "noise_share": float(noise_share)
        }
    }
}

# Имя файла JSON можно оставить общим, так как мы пишем внутрь структуру
with open(ART_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

In [97]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=best_km_labels, s=5, cmap="tab10")
plt.title(f"PCA + KMeans (best k) ({ds_name})")
plt.xlabel("PC1")
plt.ylabel("PC2")
save_path = FIG_DIR / f"{ds_name}_pca_kmeans.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close()


In [98]:
from sklearn.metrics import adjusted_rand_score

ari_scores = []

labels_ref = best_km_labels

for rs in [0, 1, 2, 3, 4]:
    km_tmp = KMeans(n_clusters=best_k, n_init=10, random_state=rs)
    lab_tmp = km_tmp.fit_predict(X)
    ari_scores.append(adjusted_rand_score(labels_ref, lab_tmp))

print("ARI по 5 запускам:", ari_scores)
print("Средний ARI:", np.mean(ari_scores))


ARI по 5 запускам: [0.999596238519046, 0.9998318322940898, 1.0, 1.0, 1.0]
Средний ARI: 0.9998856141626271


In [99]:
best_method = "KMeans"
best_config = {"k": best_k}

with open(ART_DIR / "best_configs.json", "w", encoding="utf-8") as f:
    json.dump({ds_name: best_config}, f, indent=2)

labels_df = pd.DataFrame({
    "sample_id": sample_id,
    "cluster_label": best_km_labels
})

output_csv_name = f"labels_hw07_{ds_name}.csv" 

labels_df.to_csv(LABEL_DIR / output_csv_name, index=False)
print(f"Результаты сохранены в файл: {output_csv_name}")

Результаты сохранены в файл: labels_hw07_ds3.csv
